In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("ventas_ecommerce_limpio.csv")

In [3]:
df.head()
#1. Muestra las primeras filas.

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,total_venta
0,1001,2026-07-01,Ana Lopez,Mouse,Accesorios,1,250.0,Efectivo,Cuernavaca,250.0
1,1002,2026-07-01,Luis Perez,Teclado,Accesorios,1,650.0,Tarjeta,Jiutepec,650.0
2,1003,2026-07-02,Sofia Ruiz,Audifonos,Accesorios,1,900.0,Tarjeta,Temixco,900.0
3,1004,2026-07-02,Pedro Mata,Webcam,Accesorios,1,800.0,Efectivo,Cuernavaca,800.0
4,1005,2026-07-03,Laura Diaz,Cable HDMI,Accesorios,2,180.0,Efectivo,Jiutepec,360.0


In [4]:
df.columns
#2. Revisa las columnas.

Index(['id_venta', 'fecha', 'cliente', 'producto', 'categoria', 'cantidad',
       'precio_unitario', 'metodo_pago', 'ciudad', 'total_venta'],
      dtype='object')

In [5]:
df.shape
#3. Muestra cuantas filas y columnas tiene.

(60, 10)

In [11]:
df.isnull().sum()
#4. Revisa si hay valores nulos.

id_venta           0
fecha              0
cliente            0
producto           0
categoria          0
cantidad           0
precio_unitario    0
metodo_pago        0
ciudad             0
total_venta        0
dtype: int64

In [8]:
"total_venta" in df.columns
#5. Verifica que exista la columna `total_venta`.

True

In [13]:
df["total_venta"] = df["cantidad"] * df["precio_unitario"]
#6. Verifica que `total_venta` coincida con `cantidad * precio_unitario`.

In [14]:
#El data set presenta una consistenacia en los datos bastante buena y estructurada, ademas la msiam no cuenta ocn valores
#nulos que permitira la correcta prediccion de los modelos a su vez el mismo no cuneta con errores de estandarizacion mas que
#en algunos productos

In [ ]:
## Parte 3. Variable Objetivo

In [16]:
df["venta_alta"] =  df["total_venta"].apply(lambda x: 1 if x >= 1000 else 0)

In [17]:
df["venta_alta"].value_counts()

venta_alta
1    40
0    20
Name: count, dtype: int64

In [18]:
#Por que venta_alta es la variable objetivo? Porque esta misma permiitra hacer predicciones ademas de que la misam permite clasificar
#los valores de nuestrra ventra total para poder obetner posteriores predicciones

In [19]:
## Parte 4. Variables De Entrada

In [21]:
X_Bruto =  df[["cantidad","precio_unitario","categoria","metodo_pago","ciudad"]]
#1. Crea `X`.

In [22]:
y = df["venta_alta"]
#2. Crea `y`.

In [23]:
X = pd.get_dummies(X_Bruto)
#3. Convierte variables categoricas con `pd.get_dummies()`.

In [24]:
columnas_modelo = X.columns.tolist()
#4. Guarda la lista de columnas generadas.

In [26]:
columnas_modelo
#5. Muestra las primeras filas de `X` despues de `get_dummies()`

['cantidad',
 'precio_unitario',
 'categoria_Accesorios',
 'categoria_Electronica',
 'categoria_Muebles',
 'metodo_pago_Efectivo',
 'metodo_pago_Tarjeta',
 'metodo_pago_Transferencia',
 'ciudad_Cuernavaca',
 'ciudad_Emiliano Zapata',
 'ciudad_Jiutepec',
 'ciudad_Temixco']

In [27]:
#Por que no se debe usar total_venta como variable de entrada si venta_alta se creo a partir de total_venta?
#Porque si la incluimos el modelo ya tendira la respuesta lo caul generaria algo mas cercado a memorizar un valor
#en lugar de realizar una preduccion lo que generaria que el mismso algoritmo haga "trampa"

In [29]:
## Parte 5. Entrenamiento Y Evaluacion

In [30]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [31]:
#1. Divide los datos en entrenamiento y prueba.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#2. Usa 80% entrenamiento y 20% prueba.

In [32]:
modelo = DecisionTreeClassifier(random_state=42)
#3. Usa `random_state=42`.

In [33]:
modelo.fit(X_train, y_train)
#4. Entrena el modelo.

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [34]:
predicciones =  modelo.predict(X_test)
exactitud = accuracy_score(y_test, predicciones)
#5. Genera predicciones con los datos de prueba.

In [35]:
matriz = confusion_matrix(y_test, predicciones, labels=[1,0])
#6. Calcula exactitud.

In [50]:
print("Matriz de confusión:\n", matriz, "\n")
#7. Muestra matriz de confusion.

Matriz de confusión:
 [[8 0]
 [0 4]] 



In [51]:
resultados_prueba = pd.DataFrame({"valor_real": y_test, "prediccion": predicciones})
resultados_prueba["coincide"] = resultados_prueba["valor_real"] == resultados_prueba["prediccion"]
#8. Crea una tabla llamada `resultados_prueba` con:

In [52]:
aciertos = resultados_prueba[resultados_prueba["coincide"] == True]
errores = resultados_prueba[resultados_prueba["coincide"] == False]

In [53]:
print("Aciertos:", len(aciertos))
print("Errores:", len(errores))
#9. Cuenta cuantos aciertos y cuantos errores hubo.

Aciertos: 12
Errores: 0


In [ ]:
#1. Cual fue la exactitud? Fue del 100% la exactitud
#2. Cuantos aciertos tuvo el modelo? 12 
#3. Cuantos errores tuvo el modelo? 0
#4. Que indica la matriz de confusion? como es verdadero falso falso verdadero   [[8 0] [0 4]] 
#indica 8 aciertos, 0 falsos positivos 0 falsos negativos y 4 aciertos obviamente los primeros son referentes a las veces que dijo qeu saldria una
# y ultio con respecto a qeu saldria otra, que dijo el modelo que serai falso o verdadero 
#5. Una buena exactitud significa que el modelo ya es perfecto? Explica. NOO el modelo nunca sera perfecto, al final del dia son solo algoritmos
#los mismos pueden hacer predicciones muy exactas pero sin embargo no son infalibles, por ende no son perfectos pero si muy precisos

In [ ]:
## Parte 6. Guardar Modelo Y Columnas

In [54]:
import joblib
#1. Usa `joblib`.
import os
joblib.dump(modelo, "modelo_examen_venta_alta.pkl")
#2. Guarda el modelo entrenado.
joblib.dump(columnas_modelo, "columnas_examen_modelo.pkl")
#3. Guarda la lista de columnas usadas durante el entrenamiento.

['columnas_examen_modelo.pkl']

In [55]:
archivos = ["modelo_examen_venta_alta.pkl", "columnas_examen_modelo.pkl"]

for archivo in archivos:
    if os.path.exists(archivo):
        print(f"El archivo '{archivo}' SÍ existe en la carpeta.")
    else:
        print(f"El archivo '{archivo}' NO se encuentra.")
#4. Verifica que los archivos aparezcan en tu carpeta.

El archivo 'modelo_examen_venta_alta.pkl' SÍ existe en la carpeta.
El archivo 'columnas_examen_modelo.pkl' SÍ existe en la carpeta.


In [ ]:
#1. Para que sirve guardar el modelo? Sirve para asegurar que los nuevos datos tengan exactamente las mismas variables 
#y en el mismo orden que cuando se entrenó el modelo.

#2. Para que sirve guardar las columnas del entrenamiento? Sirve para poder hacer el uso del modelo en toro lugar ademas de que nos permite
#evitra erroers para futuras cosas como por ejemplo en el caso de que agreguemos mas datos.

#3. Que problema puede aparecer si no guardas las columnas?Si el orden de las columnas cambia (por ejemplo, 
#si la columna edad intercambia posición con precio),el modelo aplicará los pesos a los datos equivocados y 
#dará predicciones completamente erróneas sin necesariamente marcar un error.

In [ ]:
## Parte 7. Ventas Nuevas

In [56]:
ventas_nuevas = pd.DataFrame([
    {"id_venta": 1, "fecha": "2023-11-05", "cliente": "Fernando Gomez", "producto": "Laptop Gaming", "categoria": "Electronica", "cantidad": 2, "precio_unitario": 1200, "metodo_pago": "Tarjeta", "ciudad": "Guadalajara"},  # Total: 2400
    {"id_venta": 2, "fecha": "2023-11-05", "cliente": "Valeria Rios", "producto": "Smart TV 65", "categoria": "Electronica", "cantidad": 1, "precio_unitario": 1550, "metodo_pago": "Transferencia", "ciudad": "Monterrey"},   # Total: 1550
    {"id_venta": 3, "fecha": "2023-11-06", "cliente": "Roberto Sanz", "producto": "Consola Videojuegos", "categoria": "Videojuegos", "cantidad": 3, "precio_unitario": 600, "metodo_pago": "Tarjeta", "ciudad": "CDMX"},        # Total: 1800
    {"id_venta": 4, "fecha": "2023-11-06", "cliente": "Diana Morales", "producto": "Refrigerador Smart", "categoria": "Linea_Blanca", "cantidad": 1, "precio_unitario": 2200, "metodo_pago": "Tarjeta", "ciudad": "Puebla"},   # Total: 2200

    {"id_venta": 5, "fecha": "2023-11-07", "cliente": "Gabriel Torres", "producto": "Mouse Inalambrico", "categoria": "Accesorios", "cantidad": 2, "precio_unitario": 25, "metodo_pago": "Efectivo", "ciudad": "Guadalajara"}, # Total: 50
    {"id_venta": 6, "fecha": "2023-11-07", "cliente": "Patricia Vega", "producto": "Funda para Tablet", "categoria": "Accesorios", "cantidad": 1, "precio_unitario": 35, "metodo_pago": "Efectivo", "ciudad": "Monterrey"},    # Total: 35
    {"id_venta": 7, "fecha": "2023-11-08", "cliente": "Hector Luna", "producto": "Cable HDMI 4K", "categoria": "Accesorios", "cantidad": 3, "precio_unitario": 15, "metodo_pago": "Efectivo", "ciudad": "CDMX"},             # Total: 45
    {"id_venta": 8, "fecha": "2023-11-08", "cliente": "Camila Navarro", "producto": "Lampara LED Smart", "categoria": "Iluminacion_Nueva", "cantidad": 2, "precio_unitario": 40, "metodo_pago": "Tarjeta", "ciudad": "Queretaro"}, # Total: 80

    {"id_venta": 9, "fecha": "2023-11-09", "cliente": "Santiago Castro", "producto": "Escritorio Ergonomico", "categoria": "Muebles_Oficina", "cantidad": 1, "precio_unitario": 980, "metodo_pago": "Tarjeta", "ciudad": "Merida"}, # Total: 980
    {"id_venta": 10, "fecha": "2023-11-09", "cliente": "Elena Mendez", "producto": "Monitor 27 Pulgadas", "categoria": "Electronica", "cantidad": 2, "precio_unitario": 510, "metodo_pago": "Tarjeta", "ciudad": "Monterrey"}  # Total: 1020
])

#Categorías nuevas: "Linea_Blanca", "Iluminacion_Nueva", "Muebles_Oficina".
#Ciudades nuevas: "Puebla", "Queretaro", "Merida".

ventas_nuevas.to_csv("examen_ventas_nuevas.csv", index=False)
print("Archivo 'examen_ventas_nuevas.csv' guardado correctamente con 10 registros.")

Archivo 'examen_ventas_nuevas.csv' guardado correctamente con 10 registros.


In [ ]:
## Parte 8. Cargar Modelo Y Predecir

In [57]:
modelo_cargado = joblib.load("modelo_examen_venta_alta.pkl")
#1. Carga `modelo_examen_venta_alta.pkl`.
columnas_cargadas = joblib.load("columnas_examen_modelo.pkl")
#2. Carga `columnas_examen_modelo.pkl`.

In [59]:
nuevas_ventas_df = pd.read_csv("examen_ventas_nuevas.csv")
#3. Carga `examen_ventas_nuevas.csv`.

In [60]:
X_nuevas = nuevas_ventas_df[["cantidad", "precio_unitario", "categoria", "metodo_pago", "ciudad"]]
#4. Selecciona las mismas variables de entrada usadas en entrenamiento.
X_nuevas_dummies = pd.get_dummies(X_nuevas)
#5. Aplica `pd.get_dummies()`.

In [61]:
X_nuevas_reindex = X_nuevas_dummies.reindex(columns=columnas_cargadas, fill_value=0)
#6. Alinea columnas con `reindex()`.

In [62]:
predicciones_nuevas = modelo_cargado.predict(X_nuevas_reindex)
#7. Genera predicciones.

In [63]:
nuevas_ventas_df["prediccion_venta_alta"] = predicciones_nuevas
nuevas_ventas_df["interpretacion"] = nuevas_ventas_df["prediccion_venta_alta"].apply(lambda x: "Venta alta" if x == 1 else "Venta no alta")

In [64]:
# 10. Guarda el resultado como:
nuevas_ventas_df.to_csv("examen_predicciones.csv", index=False)

In [65]:
nuevas_ventas_df


,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,prediccion_venta_alta,interpretacion
0,1,2023-11-05,Fernando Gomez,Laptop Gaming,Electronica,2,1200,Tarjeta,Guadalajara,1,Venta alta
1,2,2023-11-05,Valeria Rios,Smart TV 65,Electronica,1,1550,Transferencia,Monterrey,1,Venta alta
2,3,2023-11-06,Roberto Sanz,Consola Videojuegos,Videojuegos,3,600,Tarjeta,CDMX,1,Venta alta
3,4,2023-11-06,Diana Morales,Refrigerador Smart,Linea_Blanca,1,2200,Tarjeta,Puebla,1,Venta alta
4,5,2023-11-07,Gabriel Torres,Mouse Inalambrico,Accesorios,2,25,Efectivo,Guadalajara,0,Venta no alta
5,6,2023-11-07,Patricia Vega,Funda para Tablet,Accesorios,1,35,Efectivo,Monterrey,0,Venta no alta
6,7,2023-11-08,Hector Luna,Cable HDMI 4K,Accesorios,3,15,Efectivo,CDMX,0,Venta no alta
7,8,2023-11-08,Camila Navarro,Lampara LED Smart,Iluminacion_Nueva,2,40,Tarjeta,Queretaro,1,Venta alta
8,9,2023-11-09,Santiago Castro,Escritorio Ergonomico,Muebles_Oficina,1,980,Tarjeta,Merida,1,Venta alta
9,10,2023-11-09,Elena Mendez,Monitor 27 Pulgadas,Electronica,2,510,Tarjeta,Monterrey,1,Venta alta


In [ ]:
#Que podria pasar si no usas reindex antes de predecir? Podria causar errores al momento de predecir las cosas esto de forma de que 
#ciertos daos no se capetn en la calsificacion de maenr de que lo smsimso no se predicen o no se alinena lo qeu podrai causar erroes en la
#prediccion proxima de mas valores

In [70]:
## Parte 9. Auditoria Del Modelo
# 1. Calcular total_estimado
nuevas_ventas_df["total_estimado"] = nuevas_ventas_df["cantidad"] * nuevas_ventas_df["precio_unitario"]

In [71]:
# 2. Crear venta_alta_real_estimada (1 si >= 1000, 0 si < 1000)
nuevas_ventas_df["venta_alta_real_estimada"] = nuevas_ventas_df["total_estimado"].apply(lambda x: 1 if x >= 1000 else 0)

In [72]:
# 3 y 4. Comparar y crear columna coincide
nuevas_ventas_df["coincide"] = nuevas_ventas_df["prediccion_venta_alta"] == nuevas_ventas_df["venta_alta_real_estimada"]

In [73]:
# 5 y 6. Contar coincidencias y filtrar errores
coinciden_count = nuevas_ventas_df["coincide"].sum()
no_coinciden_count = (nuevas_ventas_df["coincide"] == False).sum()

print(f"Aciertos (Coinciden): {coinciden_count}")
print(f"Errores (No coinciden): {no_coinciden_count}\n")

errores_prediccion = nuevas_ventas_df[nuevas_ventas_df["coincide"] == False].copy()

Aciertos (Coinciden): 8
Errores (No coinciden): 2



In [74]:
# 7. Revisar si están cerca del límite de 1000
if not errores_prediccion.empty:
    errores_prediccion["distancia_a_1000"] = (errores_prediccion["total_estimado"] - 1000).abs()
    print("Ventas en las que falló el modelo:")
    display(errores_prediccion[["id_venta", "total_estimado", "prediccion_venta_alta", "venta_alta_real_estimada", "distancia_a_1000"]])
else:
    print("No hubo errores en este lote. Todas las predicciones coinciden perfectamente.")


Ventas en las que falló el modelo:


,id_venta,total_estimado,prediccion_venta_alta,venta_alta_real_estimada,distancia_a_1000
7,8,80,1,0,920
8,9,980,1,0,20


In [75]:
# 8. Guardar de nuevo examen_predicciones.csv con columnas de auditoría
nuevas_ventas_df.to_csv("examen_predicciones.csv", index=False)
print(" Archivo 'examen_predicciones.csv' guardado exitosamente con columnas de auditoría.")

 Archivo 'examen_predicciones.csv' guardado exitosamente con columnas de auditoría.


In [76]:
errores_prediccion

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,prediccion_venta_alta,interpretacion,total_estimado,venta_alta_real_estimada,coincide,distancia_a_1000
7,8,2023-11-08,Camila Navarro,Lampara LED Smart,Iluminacion_Nueva,2,40,Tarjeta,Queretaro,1,Venta alta,80,0,False,920
8,9,2023-11-09,Santiago Castro,Escritorio Ergonomico,Muebles_Oficina,1,980,Tarjeta,Merida,1,Venta alta,980,0,False,20


In [ ]:
1. Cuantas ventas nuevas evaluaste? 10
2. Cuantas fueron predichas como venta alta? 4
3. Cuantas fueron predichas como venta no alta? 4
4. Cuantas coincidieron con la regla manual? 4
5. Cuantas no coincidieron? 6
6. Que ventas no coincidieron? Lampara LED Smart, Escritorio Ergonomico
7. Los errores estuvieron cerca del limite de 1000? Escritorio Ergonomic
8. Que paso con la categoria nueva? La categoria nueva se analizo con exito
9. Que paso con la ciudad nueva? La ciudad tambien se analizo con exito 

In [ ]:
Nombre:Antonio Garcia Gonzalez
Grupo:9A
Materia: Extraccion de datos
Exactitud obtenida: 100
Ventas nuevas evaluadas: 10
Coincidencias: 8 
Errores: 2
Conclusion breve: Esta practica provo el modelo pude ser exportado y probado dentro de otro nuevo dataFrame, lo que permite
utilizar modelos ya entrenado en algunos nuevos